In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
import os
import json
import time
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Literal, TypedDict, Annotated, operator
from dataclasses import dataclass, asdict
import re
from datetime import datetime, timezone

In [11]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

In [15]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview", 
    api_key=GOOGLE_API_KEY,
    temperature=0.1)

llm.invoke("hi").text

'Hello! How can I help you today?'

In [47]:
class DevState(TypedDict):
    raw_input:           str
    intent_manifest:     Optional[Dict]
    compliance_rules:    Optional[Dict]
    ip_clearance:        Optional[Dict]
    architecture:        Optional[Dict]
    generated_code:      Optional[Dict]
    explainability_docs: Optional[Dict]
    security_report:     Optional[Dict]
    quality_report:      Optional[Dict]
    hitl_decisions:      Optional[List]
    audit_log:           Annotated[List[Dict], operator.add]
    drift_alerts:        Optional[List]


In [53]:
def extract_json(text: str) -> dict:
    text = re.sub(r"```(?:json)?", "", text).replace("```", "").strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM response:\n{text}")
    return json.loads(match.group())


def make_audit_entry(agent: str, summary: str, data: dict) -> dict:

    return {
        "agent":     agent,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "summary":   summary,
        "data":      data,
    }

In [54]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).isoformat()

print(timestamp)

2026-03-13T12:40:32.921372+00:00


In [55]:
INTENT_SCHEMA = """
{
  "app_type": "string — e.g. REST API, CLI tool, web app",
  "modules": [
    {
      "name": "string",
      "description": "string",
      "tech_stack": ["string"]
    }
  ],
  "constraints": {
    "security": ["string"],
    "compliance": ["string — e.g. GDPR, HIPAA"],
    "performance": ["string"],
    "ip_notes": ["string — any third-party libraries or frameworks mentioned"]
  },
  "acceptance_criteria": ["string"]
}
"""

def intent_agent(state: DevState) -> dict:
    prompt = f"""
You are an AI software architect. Convert the user's description into a
structured intent manifest. Respond ONLY with valid JSON — no explanation,
no markdown, no extra text.

User input:
{state["raw_input"]}

Return exactly this JSON schema (fill in real values, do not include comments):
{INTENT_SCHEMA}
"""

    response = llm.invoke(prompt)
    manifest = extract_json(response.text)

    audit_entry = make_audit_entry(
        agent   = "intent_agent",
        summary = f"Parsed intent for: {state['raw_input'][:80]}",
        data    = {"app_type": manifest.get("app_type"), "modules": [m["name"] for m in manifest.get("modules", [])]}
    )

    return {
        "intent_manifest": manifest,
        "audit_log":       [audit_entry],   
    }


In [56]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      None,
        "audit_log":           [],
        "drift_alerts":        None,
    }

    print("=== Running intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]   # simulate operator.add

    print("\nIntent Manifest:")
    print(json.dumps(state["intent_manifest"], indent=2))

    print("\nAudit Log:")
    print(json.dumps(state["audit_log"], indent=2))

if __name__ == "__main__":
    main()

=== Running intent_agent ===

Intent Manifest:
{
  "app_type": "web app",
  "modules": [
    {
      "name": "Authentication Module",
      "description": "User registration and login system with JWT-based session management.",
      "tech_stack": [
        "FastAPI",
        "Passlib",
        "PyJWT"
      ]
    },
    {
      "name": "Dashboard Module",
      "description": "Protected user interface for viewing account data.",
      "tech_stack": [
        "FastAPI",
        "Jinja2",
        "HTML/CSS"
      ]
    },
    {
      "name": "Data Persistence Layer",
      "description": "Relational database management for user credentials and application state.",
      "tech_stack": [
        "PostgreSQL",
        "SQLAlchemy",
        "Alembic"
      ]
    }
  ],
  "constraints": {
    "security": [
      "Password hashing with bcrypt",
      "JWT token expiration",
      "Secure cookie handling"
    ],
    "compliance": [
      "None specified"
    ],
    "performance": [
      "Asyn

In [ ]:
import re

KNOWN_LICENSES = {
    # safe for commercial use
    "fastapi":      {"license": "MIT",        "risk": "low"},
    "sqlalchemy":   {"license": "MIT",        "risk": "low"},
    "pydantic":     {"license": "MIT",        "risk": "low"},
    "uvicorn":      {"license": "BSD",        "risk": "low"},
    "postgresql":   {"license": "PostgreSQL", "risk": "low"},
    "alembic":      {"license": "MIT",        "risk": "low"},
    "passlib":      {"license": "BSD",        "risk": "low"},
    "jose":         {"license": "MIT",        "risk": "low"},
    "httpx":        {"license": "BSD",        "risk": "low"},
    # caution — copyleft
    "gpl":          {"license": "GPL",        "risk": "high"},
    "mysql":        {"license": "GPL",        "risk": "high"},
    "celery":       {"license": "BSD",        "risk": "low"},
    "redis":        {"license": "BSD",        "risk": "low"},
}

IP_SCAN_SCHEMA = """
{
  "scanned_libraries": [
    {
      "name": "string",
      "license": "string",
      "risk_level": "low | medium | high",
      "reason": "string — one sentence explanation"
    }
  ],
  "overall_risk": "low | medium | high",
  "flagged_items": ["string — only items with medium or high risk"],
  "recommendation": "string — what to do next"
}
"""

def ip_guard_agent(state: DevState) -> dict:
    manifest = state["intent_manifest"]

    raw_libs = []

    for module in manifest.get("modules", []):
        raw_libs.extend(module.get("tech_stack", []))

    raw_libs.extend(
        manifest.get("constraints", {}).get("ip_notes", [])
    )

    local_flags = []
    for lib in raw_libs:
        key = lib.lower().strip()
        if key in KNOWN_LICENSES and KNOWN_LICENSES[key]["risk"] == "high":
            local_flags.append(f"{lib} ({KNOWN_LICENSES[key]['license']} — copyleft risk)")

    prompt = f"""
You are an IP and software license compliance officer.

Review the following libraries and frameworks extracted from a software project.
For each one, identify its open source license and assess the risk for commercial use.

Libraries to scan:
{json.dumps(raw_libs, indent=2)}

Known high-risk flags detected by local scan (include these in your response):
{json.dumps(local_flags, indent=2) if local_flags else "None detected"}

Risk levels:
- low    : permissive license (MIT, BSD, Apache 2.0, PostgreSQL) — safe for commercial use
- medium : weak copyleft (LGPL, MPL) — usable but requires care
- high   : strong copyleft (GPL, AGPL) — may require open-sourcing your code

Respond ONLY with valid JSON matching this schema exactly, no explanation, no markdown:
{IP_SCAN_SCHEMA}
"""

    response = llm.invoke(prompt)
    clearance = extract_json(response.text)

    overall_risk = clearance.get("overall_risk", "unknown")

    audit_entry = make_audit_entry(
        agent   = "ip_guard_agent",
        summary = f"IP scan complete — overall risk: {overall_risk}",
        data    = {
            "libraries_scanned": len(clearance.get("scanned_libraries", [])),
            "flagged_items":     clearance.get("flagged_items", []),
            "overall_risk":      overall_risk,
        }
    )

    return {
        "ip_clearance": clearance,
        "audit_log":    [audit_entry],
    }

In [58]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # agent 1
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]
    print(json.dumps(state["intent_manifest"], indent=2))

    # agent 2
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]
    print(json.dumps(state["ip_clearance"], indent=2))

    # audit trail so far
    print("\n=== Audit Log ===")
    print(json.dumps(state["audit_log"], indent=2))

if __name__ == "__main__":
    main()

=== intent_agent ===
{
  "app_type": "web app",
  "modules": [
    {
      "name": "Authentication Module",
      "description": "User registration and login system with JWT-based session management.",
      "tech_stack": [
        "FastAPI",
        "Passlib",
        "PyJWT"
      ]
    },
    {
      "name": "Dashboard Module",
      "description": "Protected user interface for displaying account-specific data.",
      "tech_stack": [
        "FastAPI",
        "Jinja2",
        "HTML/CSS"
      ]
    },
    {
      "name": "Data Persistence Layer",
      "description": "Relational database management for user credentials and dashboard content.",
      "tech_stack": [
        "PostgreSQL",
        "SQLAlchemy",
        "Alembic"
      ]
    }
  ],
  "constraints": {
    "security": [
      "Password hashing with bcrypt",
      "JWT token expiration",
      "Secure cookie handling"
    ],
    "compliance": [
      "None specified"
    ],
    "performance": [
      "Asynchronous datab

In [ ]:

def display_hitl_summary(state: DevState):
    manifest  = state["intent_manifest"]
    clearance = state["ip_clearance"]

    print("\n" + "="*60)
    print("  HITL GATE 1 — INTENT + IP REVIEW")
    print("="*60)

    print("\n[ Intent Manifest ]")
    print(f"  App type : {manifest.get('app_type')}")
    print(f"  Modules  :")
    for m in manifest.get("modules", []):
        print(f"    - {m['name']} → {', '.join(m.get('tech_stack', []))}")
    print(f"  Security constraints  : {manifest.get('constraints', {}).get('security', [])}")
    print(f"  Compliance constraints: {manifest.get('constraints', {}).get('compliance', [])}")
    print(f"  Acceptance criteria   :")
    for ac in manifest.get("acceptance_criteria", []):
        print(f"    - {ac}")

    print("\n[ IP Clearance Report ]")
    risk = clearance.get("overall_risk", "unknown")
    risk_display = {"low": "LOW", "medium": "MEDIUM ⚠", "high": "HIGH ✗"}.get(risk, risk.upper())
    print(f"  Overall risk : {risk_display}")
    
    flagged = clearance.get("flagged_items", [])
    if flagged:
        print(f"  Flagged items:")
        for item in flagged:
            print(f"    ! {item}")
    else:
        print(f"  Flagged items: none")

    print(f"  Recommendation: {clearance.get('recommendation', 'N/A')}")

    print("\n[ Scanned Libraries ]")
    for lib in clearance.get("scanned_libraries", []):
        risk_tag = {"low": "[ok]", "medium": "[warn]", "high": "[RISK]"}.get(lib["risk_level"], "[?]")
        print(f"  {risk_tag:8s} {lib['name']:20s} {lib['license']:15s} — {lib['reason']}")

    print("\n" + "-"*60)


def get_human_decision(state: DevState) -> dict:

    display_hitl_summary(state)

    print("\nOptions:")
    print("  [A] Approve — proceed to compliance + architecture agents")
    print("  [R] Reject  — send back to intent_agent with feedback")
    print("  [M] Modify  — approve with notes (human adds constraints)")
    print()

    while True:
        choice = input("Your decision [A/R/M]: ").strip().upper()
        if choice in ("A", "R", "M"):
            break
        print("  Invalid input — please enter A, R, or M")

    feedback    = None
    extra_notes = None

    if choice == "R":
        feedback = input("Rejection reason (will be passed back to intent_agent): ").strip()

    if choice == "M":
        extra_notes = input("Additional constraints or notes to add: ").strip()

    decision = {
        "gate":        "hitl_gate_1",
        "approved":    choice in ("A", "M"),
        "choice":      choice,
        "approver":    input("Your name (for audit log): ").strip(),
        "feedback":    feedback,
        "extra_notes": extra_notes,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
    }

    return decision


def hitl_gate_1(state: DevState) -> dict:

    if not state.get("intent_manifest") or not state.get("ip_clearance"):
        raise ValueError("hitl_gate_1 requires both intent_manifest and ip_clearance in state")

    decision = get_human_decision(state)

    updated_manifest = state["intent_manifest"]

    if decision.get("extra_notes"):
        existing = updated_manifest.get("constraints", {})
        existing.setdefault("human_notes", []).append(decision["extra_notes"])
        updated_manifest["constraints"] = existing

    # if rejected, stamp the feedback onto raw_input so intent_agent can use it
    updated_raw = state["raw_input"]
    if not decision["approved"] and decision.get("feedback"):
        updated_raw = f"{state['raw_input']}\n\n[HITL feedback]: {decision['feedback']}"

    audit_entry = make_audit_entry(
        agent   = "hitl_gate_1",
        summary = f"Gate 1 decision: {decision['choice']} by {decision['approver']}",
        data    = decision,
    )

    return {
        "hitl_decisions":  (state["hitl_decisions"] or []) + [decision],
        "intent_manifest": updated_manifest,
        "raw_input":       updated_raw,      
        "audit_log":       [audit_entry],
    }


def route_after_hitl_1(state: DevState) -> str:

    last_decision = state["hitl_decisions"][-1]

    if last_decision["approved"]:
        return "compliance_agent" 
    else:
        return "intent_agent"       

In [ ]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    print("\n=== hitl_gate_1 ===")
    result = hitl_gate_1(state)
    state["hitl_decisions"]  = result["hitl_decisions"]
    state["intent_manifest"] = result["intent_manifest"]
    state["raw_input"]       = result["raw_input"]
    state["audit_log"]      += result["audit_log"]

    next_node = route_after_hitl_1(state)
    print(f"\n>> Routing to: {next_node}")

    if next_node == "intent_agent":
        print(">> Looping back — re-running intent_agent with feedback...")
        result = intent_agent(state)   
        state["intent_manifest"] = result["intent_manifest"]
        state["audit_log"] += result["audit_log"]

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")

if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Service → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML5
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Argon2 or BCrypt', 'JWT token expiration', 'HTTPS enforcement']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissive license

In [65]:
COMPLIANCE_FRAMEWORKS = {
    "gdpr": {
        "full_name": "General Data Protection Regulation",
        "triggers":  ["user data", "login", "personal data", "email", "profile", "authentication"],
        "rules": [
            "User data must be encrypted at rest and in transit",
            "Users must be able to request data deletion",
            "Explicit consent required before data collection",
            "Data breach notification within 72 hours",
            "Minimal data collection principle must be followed",
        ]
    },
    "owasp": {
        "full_name": "OWASP Top 10",
        "triggers":  ["login", "authentication", "api", "web", "dashboard", "jwt", "password"],
        "rules": [
            "Protect against SQL injection via parameterised queries",
            "Implement proper session management and token expiry",
            "Enforce HTTPS and secure cookie flags",
            "Rate limit authentication endpoints",
            "Validate and sanitise all user inputs",
        ]
    },
    "hipaa": {
        "full_name": "Health Insurance Portability and Accountability Act",
        "triggers":  ["health", "medical", "patient", "diagnosis", "prescription", "ehr"],
        "rules": [
            "PHI must be encrypted at rest and in transit",
            "Access logs must be maintained for all PHI access",
            "Role-based access control for all health data",
            "Business Associate Agreements required for third parties",
        ]
    },
    "pci_dss": {
        "full_name": "Payment Card Industry Data Security Standard",
        "triggers":  ["payment", "card", "billing", "checkout", "stripe", "transaction"],
        "rules": [
            "Card data must never be stored in plaintext",
            "Use tokenisation for all payment data",
            "Quarterly vulnerability scans required",
            "Strict access control to cardholder data",
        ]
    },
}

COMPLIANCE_SCHEMA = """
{
  "applicable_frameworks": [
    {
      "name": "string — e.g. GDPR, OWASP Top 10",
      "reason": "string — why this framework applies",
      "rules": ["string — specific rule that must be implemented"],
      "priority": "mandatory | recommended"
    }
  ],
  "consolidated_rules": [
    {
      "rule": "string — the actual requirement",
      "framework": "string — which framework it comes from",
      "implementation_hint": "string — how to implement in the tech stack"
    }
  ],
  "gaps": ["string — things in the intent manifest that may violate compliance"],
  "overall_compliance_risk": "low | medium | high"
}
"""

def detect_frameworks_locally(manifest: dict) -> list[str]:

    manifest_text = json.dumps(manifest).lower()

    triggered = []
    for fw_key, fw in COMPLIANCE_FRAMEWORKS.items():
        if any(trigger in manifest_text for trigger in fw["triggers"]):
            triggered.append(fw_key)

    return triggered


def compliance_agent(state: DevState) -> dict:
    manifest         = state["intent_manifest"]
    ip_clearance     = state["ip_clearance"]
    triggered_fws    = detect_frameworks_locally(manifest)

    framework_context = ""
    for fw_key in triggered_fws:
        fw = COMPLIANCE_FRAMEWORKS[fw_key]
        framework_context += f"\n{fw['full_name']} ({fw_key.upper()}):\n"
        for rule in fw["rules"]:
            framework_context += f"  - {rule}\n"

    prompt = f"""
You are a regulatory compliance officer reviewing a software project.

Intent Manifest:
{json.dumps(manifest, indent=2)}

IP Clearance Summary:
- Overall risk: {ip_clearance.get('overall_risk')}
- Flagged items: {ip_clearance.get('flagged_items', [])}

Locally detected applicable frameworks and their rules:
{framework_context if framework_context else "None auto-detected — use your judgment."}

Your tasks:
1. Confirm which compliance frameworks apply and why
2. List every specific rule that must be implemented given the tech stack
3. Identify any gaps or risks in the current intent manifest
4. Give each rule an implementation hint specific to the tech stack

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{COMPLIANCE_SCHEMA}
"""

    response  = llm.invoke(prompt)
    rules     = extract_json(response.text)

    risk      = rules.get("overall_compliance_risk", "unknown")
    fw_names  = [f["name"] for f in rules.get("applicable_frameworks", [])]
    gaps      = rules.get("gaps", [])

    audit_entry = make_audit_entry(
        agent   = "compliance_agent",
        summary = f"Compliance mapped — risk: {risk} | frameworks: {', '.join(fw_names)}",
        data    = {
            "frameworks_detected":   triggered_fws,
            "frameworks_confirmed":  fw_names,
            "consolidated_rules":    len(rules.get("consolidated_rules", [])),
            "gaps_found":            len(gaps),
            "overall_risk":          risk,
        }
    )

    return {
        "compliance_rules": rules,
        "audit_log":        [audit_entry],
    }

In [67]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    approved       = False          # ← THIS was missing in your version

    while not approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]   # ← ip fix from earlier too
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

            approved = True

        else:  # A
            approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return  # ← stop pipeline, don't proceed

    # ── compliance_agent only runs if approved ──────────────────────
    if approved:
        print("\n=== compliance_agent ===")
        result = compliance_agent(state)
        state["compliance_rules"] = result["compliance_rules"]
        state["audit_log"] += result["audit_log"]

        print("\n[ Applicable Frameworks ]")
        for fw in state["compliance_rules"].get("applicable_frameworks", []):
            print(f"  {fw['priority'].upper():12s} {fw['name']}")
            print(f"               {fw['reason']}")

        print("\n[ Consolidated Rules ]")
        for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
            print(f"  {i:2}. [{r['framework']}] {r['rule']}")
            print(f"      Hint: {r['implementation_hint']}")

        print("\n[ Gaps Detected ]")
        gaps = state["compliance_rules"].get("gaps", [])
        if gaps:
            for g in gaps:
                print(f"  ! {g}")
        else:
            print("  None detected")

        print(f"\n[ Overall Compliance Risk ] "
              f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── full audit trail ────────────────────────────────────────────
    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Service → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Argon2 or BCrypt', 'JWT token expiration', 'CORS policy configuration']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further legal action is required for commercial deployment.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT          

In [68]:
ARCHITECTURE_PATTERNS = {
    "layered": {
        "description": "Traditional N-tier: presentation, business logic, data",
        "best_for":    ["web apps", "dashboards", "CRUD applications"],
        "trade_offs":  {"scalability": "medium", "complexity": "low", "security": "high"}
    },
    "microservices": {
        "description": "Independent services communicating via APIs",
        "best_for":    ["large teams", "high scale", "independent deployments"],
        "trade_offs":  {"scalability": "high", "complexity": "high", "security": "medium"}
    },
    "modular_monolith": {
        "description": "Single deployable unit with strict internal module boundaries",
        "best_for":    ["small teams", "early stage", "web apps"],
        "trade_offs":  {"scalability": "medium", "complexity": "low", "security": "high"}
    },
}

ARCHITECTURE_SCHEMA = """
{
  "selected_pattern": "string — layered | microservices | modular_monolith",
  "pattern_rationale": "string — why this pattern was chosen",
  "layers": [
    {
      "name": "string — e.g. API Layer, Auth Layer, Data Layer",
      "responsibility": "string — what this layer does",
      "components": ["string — specific classes, services or modules"],
      "tech": ["string — specific libraries or tools"],
      "compliance_controls": ["string — which compliance rules this layer satisfies"]
    }
  ],
  "infrastructure": {
    "database":    "string",
    "cache":       "string",
    "tls":         "string",
    "rate_limiter":"string",
    "audit_store": "string"
  },
  "security_controls": ["string — specific security measures built into the architecture"],
  "trade_off_matrix": {
    "scalability":  "low | medium | high",
    "complexity":   "low | medium | high",
    "security":     "low | medium | high",
    "compliance_fit":"low | medium | high"
  },
  "gaps_addressed": ["string — which compliance gaps from previous agent are now covered"],
  "residual_risks": ["string — anything still not addressed"]
}
"""

def architecture_agent(state: DevState) -> dict:
    manifest         = state["intent_manifest"]
    compliance       = state["compliance_rules"]
    ip_clearance     = state["ip_clearance"]

    # pull compliance gaps so arch agent explicitly addresses them
    gaps             = compliance.get("gaps", [])
    consolidated     = compliance.get("consolidated_rules", [])
    frameworks       = [f["name"] for f in compliance.get("applicable_frameworks", [])]

    # build pattern context for the prompt
    pattern_context  = ""
    for name, pattern in ARCHITECTURE_PATTERNS.items():
        pattern_context += f"\n{name}:\n"
        pattern_context += f"  Description : {pattern['description']}\n"
        pattern_context += f"  Best for    : {', '.join(pattern['best_for'])}\n"
        pattern_context += f"  Trade-offs  : {pattern['trade_offs']}\n"

    prompt = f"""
You are a senior software architect designing a production-grade system.

Intent Manifest:
{json.dumps(manifest, indent=2)}

Compliance Frameworks Active: {', '.join(frameworks)}

Compliance Rules to Satisfy:
{json.dumps(consolidated, indent=2)}

Compliance Gaps That Must Be Addressed in Architecture:
{json.dumps(gaps, indent=2)}

IP Clearance:
- Overall risk  : {ip_clearance.get('overall_risk')}
- Flagged items : {ip_clearance.get('flagged_items', [])}

Available Architecture Patterns:
{pattern_context}

Your tasks:
1. Select the most appropriate architecture pattern given the app type,
   team size implied by the manifest, and compliance requirements
2. Define each layer with its responsibilities, components, and tech
3. Map every compliance rule to the layer that satisfies it
4. Address every compliance gap explicitly in either a layer or infrastructure
5. Score the architecture on the trade-off matrix
6. List any residual risks that could not be fully addressed

Respond ONLY with valid JSON matching this schema, no explanation, no markdown:
{ARCHITECTURE_SCHEMA}
"""

    response = llm.invoke(prompt)
    arch     = extract_json(response.text)

    pattern  = arch.get("selected_pattern", "unknown")
    gaps_addressed = arch.get("gaps_addressed", [])
    residual = arch.get("residual_risks", [])

    audit_entry = make_audit_entry(
        agent   = "architecture_agent",
        summary = f"Architecture synthesised — pattern: {pattern} | gaps addressed: {len(gaps_addressed)}",
        data    = {
            "pattern":          pattern,
            "layers":           [l["name"] for l in arch.get("layers", [])],
            "gaps_addressed":   gaps_addressed,
            "residual_risks":   residual,
            "trade_off_matrix": arch.get("trade_off_matrix", {}),
        }
    )

    return {
        "architecture": arch,
        "audit_log":    [audit_entry],
    }

In [69]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    approved       = False          # ← THIS was missing in your version

    while not approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]   # ← ip fix from earlier too
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

            approved = True

        else:  # A
            approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return  # ← stop pipeline, don't proceed

    # ── compliance_agent only runs if approved ──────────────────────
    if approved:
        print("\n=== compliance_agent ===")
        result = compliance_agent(state)
        state["compliance_rules"] = result["compliance_rules"]
        state["audit_log"] += result["audit_log"]

        print("\n[ Applicable Frameworks ]")
        for fw in state["compliance_rules"].get("applicable_frameworks", []):
            print(f"  {fw['priority'].upper():12s} {fw['name']}")
            print(f"               {fw['reason']}")

        print("\n[ Consolidated Rules ]")
        for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
            print(f"  {i:2}. [{r['framework']}] {r['rule']}")
            print(f"      Hint: {r['implementation_hint']}")

        print("\n[ Gaps Detected ]")
        gaps = state["compliance_rules"].get("gaps", [])
        if gaps:
            for g in gaps:
                print(f"  ! {g}")
        else:
            print("  None detected")

        print(f"\n[ Overall Compliance Risk ] "
              f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        bar = {"low": "░░░░░░", "medium": "███░░░", "high": "██████"}.get(v, "?")
        print(f"  {k:20s} {bar}  {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with Bcrypt', 'JWT token expiration', 'CORS policy configuration']
  Compliance constraints: ['None']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further action is required for compliance.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissive license allowing comme

In [72]:
def display_hitl2_summary(state: DevState):
    arch       = state["architecture"]
    compliance = state["compliance_rules"]

    print("\n" + "="*60)
    print("  HITL GATE 2 — ARCHITECTURE + COMPLIANCE REVIEW")
    print("="*60)

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern', '').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        bar = {"low": "░░░░░░", "medium": "███░░░", "high": "██████"}.get(v, "?")
        print(f"  {k:20s} {bar}  {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"  {layer['name']}")
        print(f"    Tech      : {', '.join(layer.get('tech', []))}")
        print(f"    Satisfies : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Compliance Gaps Addressed ]")
    original_gaps = compliance.get("gaps", [])
    gaps_addressed = arch.get("gaps_addressed", [])
    
    # cross check — flag any gap not addressed
    unresolved = []
    for gap in original_gaps:
        matched = any(
            any(word in addressed.lower() for word in gap.lower().split()[:3])
            for addressed in gaps_addressed
        )
        if matched:
            print(f"  [ok] {gap}")
        else:
            print(f"  [!!] UNRESOLVED: {gap}")
            unresolved.append(gap)

    if arch.get("residual_risks"):
        print("\n[ Residual Risks — Human Sign-off Required ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    if unresolved:
        print("\n[ WARNING — Unresolved Gaps Detected ]")
        for u in unresolved:
            print(f"  [!!] {u}")

    print("\n" + "-"*60)


def get_human_decision_2(state: DevState) -> dict:
    display_hitl2_summary(state)

    arch     = state["architecture"]
    residual = arch.get("residual_risks", [])

    # auto warn if residual risks exist
    if residual:
        print(f"\n  NOTE: {len(residual)} residual risk(s) require acknowledgement:")
        for r in residual:
            print(f"    [!] {r}")
        print()

    print("Options:")
    print("  [A] Approve — proceed to codegen + explainability agents")
    print("  [R] Reject  — send back to architecture_agent with feedback")
    print("  [M] Modify  — approve with additional arch constraints")
    print()

    while True:
        choice = input("Your decision [A/R/M]: ").strip().upper()
        if choice in ("A", "R", "M"):
            break
        print("  Invalid input — please enter A, R, or M")

    feedback    = None
    extra_notes = None
    risk_acknowledged = False

    if choice == "R":
        feedback = input("Rejection reason (passed back to architecture_agent): ").strip()

    if choice == "M":
        extra_notes = input("Additional architecture constraints: ").strip()

    # force acknowledgement if residual risks exist
    if residual and choice in ("A", "M"):
        print("\n  Residual risks must be explicitly acknowledged before approval.")
        ack = input("  Type ACKNOWLEDGE to confirm you accept these risks: ").strip().upper()
        risk_acknowledged = ack == "ACKNOWLEDGE"
        if not risk_acknowledged:
            print("  Acknowledgement not confirmed — defaulting to rejection.")
            choice   = "R"
            feedback = "Residual risks not acknowledged by reviewer."

    decision = {
        "gate":               "hitl_gate_2",
        "approved":           choice in ("A", "M"),
        "choice":             choice,
        "approver":           input("Your name (for audit log): ").strip(),
        "feedback":           feedback,
        "extra_notes":        extra_notes,
        "risk_acknowledged":  risk_acknowledged,
        "residual_risks_seen": arch.get("residual_risks", []),
        "timestamp":          datetime.now(timezone.utc).isoformat(),
    }

    return decision


def hitl_gate_2(state: DevState) -> dict:

    if not state.get("architecture") or not state.get("compliance_rules"):
        raise ValueError("hitl_gate_2 requires architecture and compliance_rules in state")

    decision = get_human_decision_2(state)

    updated_arch = state["architecture"]

    # merge extra constraints into architecture if modified
    if decision.get("extra_notes"):
        updated_arch.setdefault("human_constraints", []).append(decision["extra_notes"])

    # if rejected stamp feedback for architecture_agent rerun
    updated_raw = state["raw_input"]
    if not decision["approved"] and decision.get("feedback"):
        updated_raw = (
            f"{state['raw_input']}\n\n"
            f"[HITL Gate 2 feedback]: {decision['feedback']}"
        )

    audit_entry = make_audit_entry(
        agent   = "hitl_gate_2",
        summary = f"Gate 2 decision: {decision['choice']} by {decision['approver']} | risks acknowledged: {decision['risk_acknowledged']}",
        data    = decision,
    )

    return {
        "hitl_decisions": (state["hitl_decisions"] or []) + [decision],
        "architecture":   updated_arch,
        "raw_input":      updated_raw,
        "audit_log":      [audit_entry],
    }


def route_after_hitl_2(state: DevState) -> str:
    last = state["hitl_decisions"][-1]

    if last["gate"] != "hitl_gate_2":
        raise ValueError("route_after_hitl_2 called but last decision is not from gate 2")

    if last["approved"]:
        return "codegen_agent"
    else:
        return "architecture_agent"

In [73]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      [],
        "audit_log":           [],
        "drift_alerts":        None,
    }

    # ── agent 1 ──
    print("=== intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]

    # ── agent 2 ──
    print("\n=== ip_guard_agent ===")
    result = ip_guard_agent(state)
    state["ip_clearance"] = result["ip_clearance"]
    state["audit_log"] += result["audit_log"]

    # ── HITL gate 1 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    approved       = False          # ← THIS was missing in your version

    while not approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_1 (attempt {iteration}) ===")

        result = hitl_gate_1(state)
        state["hitl_decisions"]  = result["hitl_decisions"]
        state["intent_manifest"] = result["intent_manifest"]
        state["raw_input"]       = result["raw_input"]
        state["audit_log"]      += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running intent_agent with feedback...")
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            print(">> Re-running ip_guard_agent on updated manifest...")
            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]   # ← ip fix from earlier too
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — applying: {last['extra_notes']}")
            state["raw_input"] = (
                f"{state['raw_input']}\n\n"
                f"[Human modification]: {last['extra_notes']}\n"
                f"Please update the intent manifest to reflect this change exactly."
            )
            result = intent_agent(state)
            state["intent_manifest"] = result["intent_manifest"]
            state["audit_log"] += result["audit_log"]

            result = ip_guard_agent(state)
            state["ip_clearance"] = result["ip_clearance"]
            state["audit_log"] += result["audit_log"]

            print("\n>> Updated manifest modules:")
            for m in state["intent_manifest"].get("modules", []):
                print(f"   - {m['name']} → {', '.join(m.get('tech_stack', []))}")

            approved = True

        else:  # A
            approved = True
            print("\n>> Approved — proceeding to compliance_agent")

    if not approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return  # ← stop pipeline, don't proceed

    # ── compliance_agent only runs if approved ──────────────────────
    if approved:
        print("\n=== compliance_agent ===")
        result = compliance_agent(state)
        state["compliance_rules"] = result["compliance_rules"]
        state["audit_log"] += result["audit_log"]

        print("\n[ Applicable Frameworks ]")
        for fw in state["compliance_rules"].get("applicable_frameworks", []):
            print(f"  {fw['priority'].upper():12s} {fw['name']}")
            print(f"               {fw['reason']}")

        print("\n[ Consolidated Rules ]")
        for i, r in enumerate(state["compliance_rules"].get("consolidated_rules", []), 1):
            print(f"  {i:2}. [{r['framework']}] {r['rule']}")
            print(f"      Hint: {r['implementation_hint']}")

        print("\n[ Gaps Detected ]")
        gaps = state["compliance_rules"].get("gaps", [])
        if gaps:
            for g in gaps:
                print(f"  ! {g}")
        else:
            print("  None detected")

        print(f"\n[ Overall Compliance Risk ] "
              f"{state['compliance_rules'].get('overall_compliance_risk', '').upper()}")

    # ── architecture_agent ──────────────────────────────────────────
    print("\n=== architecture_agent ===")
    result = architecture_agent(state)
    state["architecture"] = result["architecture"]
    state["audit_log"] += result["audit_log"]

    arch = state["architecture"]

    print(f"\n[ Selected Pattern ]  {arch.get('selected_pattern').upper()}")
    print(f"  Rationale: {arch.get('pattern_rationale')}")

    print("\n[ Trade-off Matrix ]")
    for k, v in arch.get("trade_off_matrix", {}).items():
        bar = {"low": "░░░░░░", "medium": "███░░░", "high": "██████"}.get(v, "?")
        print(f"  {k:20s} {bar}  {v.upper()}")

    print("\n[ Layers ]")
    for layer in arch.get("layers", []):
        print(f"\n  {layer['name']}")
        print(f"    Responsibility : {layer['responsibility']}")
        print(f"    Tech           : {', '.join(layer.get('tech', []))}")
        print(f"    Components     : {', '.join(layer.get('components', []))}")
        if layer.get("compliance_controls"):
            print(f"    Satisfies      : {', '.join(layer.get('compliance_controls', []))}")

    print("\n[ Infrastructure ]")
    for k, v in arch.get("infrastructure", {}).items():
        print(f"  {k:15s} : {v}")

    print("\n[ Security Controls ]")
    for sc in arch.get("security_controls", []):
        print(f"  + {sc}")

    print("\n[ Compliance Gaps Addressed ]")
    for g in arch.get("gaps_addressed", []):
        print(f"  [resolved] {g}")

    if arch.get("residual_risks"):
        print("\n[ Residual Risks ]")
        for r in arch.get("residual_risks", []):
            print(f"  [!] {r}")

    # ── HITL gate 2 — loop until approved ──────────────────────────
    max_iterations = 3
    iteration      = 0
    gate2_approved = False

    while not gate2_approved and iteration < max_iterations:
        iteration += 1
        print(f"\n=== hitl_gate_2 (attempt {iteration}) ===")

        result = hitl_gate_2(state)
        state["hitl_decisions"] = result["hitl_decisions"]
        state["architecture"]   = result["architecture"]
        state["raw_input"]      = result["raw_input"]
        state["audit_log"]     += result["audit_log"]

        last = state["hitl_decisions"][-1]

        if last["choice"] == "R":
            print(f"\n>> Rejected — reason: {last['feedback']}")
            print(">> Re-running architecture_agent with feedback...")

            result = architecture_agent(state)
            state["architecture"] = result["architecture"]
            state["audit_log"]   += result["audit_log"]

            print("\n>> Updated layers:")
            for layer in state["architecture"].get("layers", []):
                print(f"   - {layer['name']} → {', '.join(layer.get('tech', []))}")

        elif last["choice"] == "M" and last.get("extra_notes"):
            print(f"\n>> Modified — constraint added: {last['extra_notes']}")
            gate2_approved = True

        else:  # A
            gate2_approved = True
            print("\n>> Approved — proceeding to codegen + explainability agents")

    if not gate2_approved:
        print(f"\n>> Max iterations ({max_iterations}) reached — escalating")
        return

    print("\n=== Full Audit Log ===")
    for entry in state["audit_log"]:
        print(f"  [{entry['timestamp']}] {entry['agent']:20s} — {entry['summary']}")


if __name__ == "__main__":
    main()

=== intent_agent ===

=== ip_guard_agent ===

=== hitl_gate_1 (attempt 1) ===

  HITL GATE 1 — INTENT + IP REVIEW

[ Intent Manifest ]
  App type : web app
  Modules  :
    - Authentication Module → FastAPI, Passlib, PyJWT
    - Dashboard Module → FastAPI, Jinja2, HTML/CSS
    - Data Persistence Layer → PostgreSQL, SQLAlchemy, Alembic
  Security constraints  : ['Password hashing with bcrypt', 'JWT token expiration', 'Secure cookie handling']
  Compliance constraints: ['None specified']
  Acceptance criteria   :
    - User can register a new account
    - User can log in and receive a session token
    - Dashboard is inaccessible to unauthenticated users
    - Data is successfully persisted in PostgreSQL

[ IP Clearance Report ]
  Overall risk : LOW
  Flagged items: none
  Recommendation: All identified libraries are under permissive licenses; no further action is required for compliance.

[ Scanned Libraries ]
  [ok]     FastAPI              MIT             — Permissive license allowin